In [7]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

files = ['chameleon.wav', 'cantaloupe.wav', 'rockit.wav']
titles = {
    'chameleon.wav': 'Chameleon (1970s)',
    'cantaloupe.wav': 'Cantaloupe Island (1960s)',
    'rockit.wav': 'Rockit (1983)'
}

audio_data = {}
sr_data = {}
onset_times_data = {}
beat_times_data = {}

hop_length = 512

for file in files:
    path = f'../data/audio/{file}'
    
    y, sr = librosa.load(path, sr=None)
    audio_data[file] = y
    sr_data[file] = sr
    
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
    
    onset_frames = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr, hop_length=hop_length, backtrack=True)
    onset_times_data[file] = librosa.frames_to_time(onset_frames, sr=sr, hop_length=hop_length)
    
    # Визначаємо стабільні долі (Beats) з автоматичним визначенням темпу
    tempo, beat_frames = librosa.beat.beat_track( onset_envelope=onset_env, sr=sr, hop_length=hop_length)
    beat_times_data[file] = librosa.frames_to_time(beat_frames, sr=sr, hop_length=hop_length)
    
    print(f"Loaded and verified: {titles[file]}")

Loaded and verified: Chameleon (1970s)
Loaded and verified: Cantaloupe Island (1960s)
Loaded and verified: Rockit (1983)


In [8]:
tolerance = 125.0  # мс

for file in files:
    onsets = onset_times_data[file]
    beats = beat_times_data[file]
    
    diffs = []
    for ons in onsets:
        # знаходимо найближчий біт
        best_beat = beats[np.argmin(np.abs(beats - ons))]
        dev = (ons - best_beat) * 1000.0  # в мс
        
        if np.abs(dev) <= tolerance:
            diffs.append(dev)
            
    print(file)
    print("onsets count:", len(diffs))
    print("mean:", np.mean(diffs))
    print("std:", np.std(diffs))
    print("first 10:", diffs[:10])
    print()

chameleon.wav
onsets count: 169
mean: 5.564545344765071
std: 67.92174353809986
first 10: [np.float64(-23.21995464852611), np.float64(-11.609977324263054), np.float64(-23.21995464852611), np.float64(116.09977324263032), np.float64(-23.21995464852611), np.float64(116.0997732426301), np.float64(-23.21995464852611), np.float64(-46.43990929705222), np.float64(-46.43990929705177), np.float64(-23.219954648525665)]

cantaloupe.wav
onsets count: 134
mean: -20.18749788472569
std: 40.76292644847339
first 10: [np.float64(-58.04988662131519), np.float64(-34.82993197278911), np.float64(-46.43990929705222), np.float64(-34.82993197278916), np.float64(-23.21995464852611), np.float64(-46.43990929705222), np.float64(-46.43990929705222), np.float64(-23.21995464852611), np.float64(-23.21995464852611), np.float64(-23.219954648525665)]

rockit.wav
onsets count: 78
mean: 29.47148090005224
std: 68.55188295096102
first 10: [np.float64(-23.21995464852608), np.float64(116.09977324263039), np.float64(-58.049886621